In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from xgboost import XGBRegressor

def haversine_np(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c

DATA_PATH = r"c:\Users\ADMIN\Documents\Storm path predictor\Datasets\data.csv"
MODEL_DIR = r"c:\Users\ADMIN\Documents\Storm path predictor\src\train\Xgboost\models"

df = pd.read_csv(DATA_PATH, low_memory=False)
df['time'] = pd.to_datetime(df['time'])
df = df[df['time'].dt.year >= 1980].copy()
df = df.sort_values(by=['international_id', 'time'])

# Calculate the time step inside the storm (1st observation, 2nd, etc.)
df['storm_step'] = df.groupby('international_id').cumcount() + 1

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['international_id']))
df_test = df.iloc[test_idx].copy()

def feature_engineering(df):
    df = df.copy()
    df['month'] = df['time'].dt.month
    df['day'] = df['time'].dt.day
    df['hour'] = df['time'].dt.hour
    
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['day_sin'] = np.sin(2 * np.pi * df['day'] / 31)
    df['day_cos'] = np.cos(2 * np.pi * df['day'] / 31)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    
    horizons = {'6h': -1, '12h': -2, '18h': -3, '24h': -4}
    for h_name, shift_val in horizons.items():
        df[f'future_lat_{h_name}'] = df.groupby('international_id')['lat'].shift(shift_val)
        df[f'future_lon_{h_name}'] = df.groupby('international_id')['lon'].shift(shift_val)
        df[f'delta_lat_{h_name}'] = df[f'future_lat_{h_name}'] - df['lat']
        df[f'delta_lon_{h_name}'] = df[f'future_lon_{h_name}'] - df['lon']
        df.drop(columns=[f'future_lat_{h_name}', f'future_lon_{h_name}'], inplace=True)
        
    lag_props = ['lat', 'lon', 'max_wind_kt', 'dir_50kt', 'rad_50kt_long_nm', 'rad_50kt_short_nm', 
                 'dir_30kt', 'rad_30kt_long_nm', 'rad_30kt_short_nm']
    lags = {'6h': 1, '12h': 2, '18h': 3, '24h': 4}
    for col in lag_props:
        if col in df.columns:
            for l_name, shift_val in lags.items():
                df[f'{col}_lag_{l_name}'] = df.groupby('international_id')[col].shift(shift_val)
                if col in ['lat', 'lon']:
                    df[f'past_delta_{col}_{l_name}'] = df[col] - df[f'{col}_lag_{l_name}']
                
    target_cols = []
    for h in ['6h', '12h', '18h', '24h']:
        target_cols.extend([f'delta_lat_{h}', f'delta_lon_{h}'])
    
    df.dropna(subset=target_cols, inplace=True)
    
    cols_to_drop = ['tc_number', 'name', 'grade_name', 'revision_date', 'year', 'time_diff_hours', 'time', 'month', 'day', 'hour', 'flag_last']
    cols_to_drop = [c for c in cols_to_drop if c in df.columns]
    df.drop(columns=cols_to_drop, inplace=True)

    object_cols = df.select_dtypes(include=['object']).columns
    object_cols = [c for c in object_cols if c != 'international_id']
    df.drop(columns=object_cols, inplace=True)
    
    df = df.apply(pd.to_numeric, errors='coerce')
    return df, target_cols

df_test, target_cols = feature_engineering(df_test)
X_test = df_test.drop(columns=target_cols + ['international_id', 'storm_step'])
y_test = df_test[target_cols]

horizons = ['6h', '12h', '18h', '24h']
losses = {h: [] for h in horizons}

for h in horizons:
    lat_target = f'delta_lat_{h}'
    lon_target = f'delta_lon_{h}'
    
    model_lat = joblib.load(os.path.join(MODEL_DIR, f'xgboost_{lat_target}.joblib'))
    model_lon = joblib.load(os.path.join(MODEL_DIR, f'xgboost_{lon_target}.joblib'))
    
    pred_lat = model_lat.predict(X_test)
    pred_lon = model_lon.predict(X_test)
    
    current_lat = X_test['lat'].values
    current_lon = X_test['lon'].values
    
    true_final_lat = current_lat + y_test[lat_target].values
    true_final_lon = current_lon + y_test[lon_target].values
    
    pred_final_lat = current_lat + pred_lat
    pred_final_lon = current_lon + pred_lon
    
    distance_error_km = haversine_np(true_final_lon, true_final_lat, pred_final_lon, pred_final_lat)
    
    df_result = pd.DataFrame({
        'storm_step': df_test['storm_step'],
        'error_km': distance_error_km
    })
    
    avg_error_per_step = df_result.groupby('storm_step')['error_km'].mean().reset_index()
    losses[h] = avg_error_per_step

plt.figure(figsize=(15, 10))
for i, h in enumerate(horizons, 1):
    plt.subplot(2, 2, i)
    plt.plot(losses[h]['storm_step'], losses[h]['error_km'], marker='o', linestyle='-')
    plt.title(f'Average Loss (Distance Error) vs Storm Step for {h} Model')
    plt.xlabel('Storm Step (Observation Index)')
    plt.ylabel('Average Error (km)')
    plt.grid(True)

plt.tight_layout()
plt.show()
